In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 26. Unsupervised Learning: DBSCAN (Density-Based Spatial Clustering)

## Algorithm Category
**Type**: Unsupervised Learning - Clustering  
**Complexity**: Medium  
**Use Case**: Density-based clustering that finds clusters of arbitrary shape and handles noise

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand DBSCAN algorithm and density-based clustering concepts
- Implement DBSCAN clustering
- Understand core points, border points, and noise points
- Tune hyperparameters (eps, min_samples)
- Handle noise/outliers automatically
- Apply DBSCAN to real-world problems

## Historical Context

DBSCAN was developed by Ester, Kriegel, Sander, and Xu in 1996:
- Ester, M., et al. (1996): "A density-based algorithm for discovering clusters in large spatial databases"
- Handles clusters of arbitrary shape
- Automatically identifies noise points

**Key Papers/References:**
- Ester, M., et al. (1996). "A density-based algorithm for discovering clusters in large spatial databases"

## When to Use DBSCAN

DBSCAN is appropriate when:
- You don't know the number of clusters
- Clusters have arbitrary shapes (non-spherical)
- You need to identify noise/outliers
- Clusters have varying densities
- You want to handle outliers automatically
- Working with spatial data

## Theory & Mechanics

### Mathematical Foundation

DBSCAN groups points that are closely packed together (density-connected).

**Key Concepts:**

1. **eps (ε)**: Maximum distance between two samples to be considered neighbors
2. **min_samples**: Minimum number of samples in a neighborhood to form a core point
3. **Core Point**: Point with at least min_samples neighbors within eps distance
4. **Border Point**: Point that is reachable from a core point but has fewer than min_samples neighbors
5. **Noise Point**: Point that is neither core nor border

**Algorithm Steps:**

1. **Initialize**: Mark all points as unvisited
2. **Select**: Pick an unvisited point p
3. **Check**: If p has at least min_samples neighbors within eps:
   - Create new cluster C
   - Add p to C
   - Mark p as visited
   - Add all density-reachable points to C
4. **Repeat**: Continue until all points visited

**Density-Reachable:**
- Point q is density-reachable from p if there's a chain of points where each is within eps of the next
- All points in a cluster are density-connected

### How It Works

1. **Find Core Points**: Points with enough neighbors
2. **Form Clusters**: Connect density-reachable core points
3. **Add Border Points**: Assign border points to nearest cluster
4. **Mark Noise**: Remaining points are noise/outliers

### Key Hyperparameters

- **eps**: Maximum distance between samples (most important)
- **min_samples**: Minimum samples in neighborhood
- **metric**: Distance metric ('euclidean', 'manhattan', etc.)

### Advantages

- No need to specify number of clusters
- Handles clusters of arbitrary shape
- Automatically identifies noise/outliers
- Robust to outliers
- Works well with varying cluster densities
- Only two parameters to tune

### Limitations

- Sensitive to eps parameter
- Struggles with clusters of very different densities
- Not suitable for high-dimensional data (curse of dimensionality)
- Border points may be assigned to wrong cluster
- Can be slow for large datasets


## Implementation

Let's implement DBSCAN clustering.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    make_moons,  # Generate moon-shaped clusters (non-spherical, for testing)
    make_circles,  # Generate circular clusters (non-spherical)
    make_blobs  # Generate blob-shaped clusters (spherical)
)
from sklearn.cluster import DBSCAN  # DBSCAN density-based clustering
from sklearn.preprocessing import StandardScaler  # Feature scaling
from sklearn.metrics import silhouette_score  # Calculate silhouette score (cluster quality metric)
from sklearn.neighbors import NearestNeighbors  # Find nearest neighbors (for eps estimation)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.unsupervised import (
    dbscan_cluster,  # DBSCAN clustering wrapper function
    evaluate_clustering  # Evaluate clustering quality (silhouette score)
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# GENERATING NON-SPHERICAL DATASET: Testing DBSCAN
# ============================================

# make_moons() generates two crescent-shaped clusters (non-spherical)
# This is perfect for testing DBSCAN, which can handle non-spherical clusters
# K-Means would struggle with this shape!

# make_moons() parameters:
# n_samples=300: Number of data points to generate
# noise=0.1: Amount of random noise (0 = perfect moons, higher = more noise)
# random_state=42: Ensures reproducible results
X, y_true = make_moons(n_samples=300, noise=0.1, random_state=42)
# Returns:
# - X: Feature values (300 samples × 2 features)
# - y_true: True cluster labels (for comparison - we won't use these for clustering!)

print(f"Dataset Shape: {X.shape}")  # Output: (300, 2) - 300 points, 2 features
print(f"True number of clusters: {len(np.unique(y_true))}")  # Output: 2 clusters

# ============================================
# APPLYING DBSCAN CLUSTERING
# ============================================

# DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
# Finds clusters based on density (points close together)
# Automatically identifies noise points (outliers)

# DBSCAN parameters:
# eps=0.3: Maximum distance between two samples to be considered neighbors
#   - Smaller = tighter clusters, more noise points
#   - Larger = looser clusters, fewer noise points
#   - This is the most important parameter!
#
# min_samples=10: Minimum number of samples in a neighborhood to form a core point
#   - Smaller = more clusters, more noise
#   - Larger = fewer clusters, less noise
dbscan = DBSCAN(eps=0.3, min_samples=10)

# fit_predict() trains the model and returns cluster assignments
y_pred = dbscan.fit_predict(X)
# Returns: array of cluster labels
#   - Positive numbers (0, 1, 2, ...): Cluster assignments
#   - -1: Noise points (outliers that don't belong to any cluster)

# ============================================
# COUNTING CLUSTERS AND NOISE
# ============================================

# Count number of clusters (excluding noise label -1)
n_clusters = len(set(y_pred)) - (1 if -1 in y_pred else 0)
# set(y_pred): Unique cluster labels (e.g., {0, 1, -1})
# -1 in y_pred: Check if there are noise points
# Subtract 1 if noise exists (because -1 is not a cluster)

# Count number of noise points
n_noise = list(y_pred).count(-1)  # Count how many times -1 appears

print(f"\nDBSCAN Results:")
print(f"  Number of clusters: {n_clusters}")  # Number of clusters found
print(f"  Number of noise points: {n_noise}")  # Number of outliers
print(f"  eps: {dbscan.eps}")  # Epsilon parameter used
print(f"  min_samples: {dbscan.min_samples}")  # Min samples parameter used

# ============================================
# VISUALIZING CLUSTERING RESULTS
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Width=12 inches, height=5 inches

# Subplot 1: True clusters (ground truth)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot colored by true labels
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title('True Clusters')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: DBSCAN discovered clusters
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Get unique cluster labels (including -1 for noise)
unique_labels = set(y_pred)  # Set of unique labels (e.g., {0, 1, -1})

# Create colors for each cluster (including noise)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
# plt.cm.Spectral: Color map (rainbow colors)
# np.linspace(0, 1, len(unique_labels)): Evenly spaced values from 0 to 1
# This creates different colors for each cluster

# Plot each cluster separately (including noise)
for k, col in zip(unique_labels, colors):
    # k: Cluster label (0, 1, 2, ... or -1 for noise)
    # col: Color for this cluster
    
    if k == -1:
        # Noise points: special handling
        col = 'black'  # Black color for noise
        marker = 'x'  # X marker for noise points
        label = 'Noise'  # Label for legend
    else:
        # Regular clusters
        marker = 'o'  # Circle marker
        label = f'Cluster {k}'  # Label: "Cluster 0", "Cluster 1", etc.
    
    # Create boolean mask for samples in this cluster
    class_member_mask = (y_pred == k)  # True for samples in cluster k
    
    # Get coordinates of samples in this cluster
    xy = X[class_member_mask]  # Feature values for samples in cluster k
    
    # Scatter plot for this cluster
    plt.scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=50, alpha=0.7, label=label)
    # xy[:, 0]: First feature, xy[:, 1]: Second feature
    # c=[col]: Color for this cluster
    # marker: Shape (circle or X)
    # s=50: Point size
    # alpha=0.7: Semi-transparent
    # label: Label for legend

plt.title('DBSCAN Clustering')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.legend()  # Show legend (cluster names)
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# Interpretation:
# - Compare left plot (true clusters) with right plot (discovered clusters)
# - DBSCAN can find non-spherical clusters (unlike K-Means)
# - Noise points (black X's) are outliers that don't belong to any cluster
# - If clusters match true labels well, DBSCAN found the right structure


## Finding Optimal eps Parameter

Let's use the k-distance graph to find optimal eps.


In [ ]:
# ============================================
# FINDING OPTIMAL EPS: K-Distance Graph Method
# ============================================

# eps is the most important DBSCAN parameter, but it's hard to choose
# The k-distance graph method helps find a good eps value
# Idea: Plot distance to k-th nearest neighbor for each point
# The "knee" (elbow) in the graph suggests a good eps value

# Set min_samples (same as DBSCAN parameter)
min_samples = 10  # Minimum samples in neighborhood

# ============================================
# FINDING K-TH NEAREST NEIGHBORS
# ============================================

# NearestNeighbors finds nearest neighbors for each point
neighbors = NearestNeighbors(n_neighbors=min_samples)
# n_neighbors=min_samples: Find k-th nearest neighbor (where k = min_samples)

# Fit to data (builds search structure)
neighbors_fit = neighbors.fit(X)

# Find k nearest neighbors for each point
distances, indices = neighbors_fit.kneighbors(X)
# distances: Distances to k nearest neighbors (n_samples × k)
#   - Each row: distances from one point to its k nearest neighbors
# indices: Indices of k nearest neighbors (n_samples × k)

# ============================================
# EXTRACTING K-TH NEAREST NEIGHBOR DISTANCES
# ============================================

# Sort distances for each point (ascending order)
distances = np.sort(distances, axis=0)
# axis=0: Sort along rows (for each point, sort its neighbor distances)

# Extract k-th nearest neighbor distance (last column, since we sorted)
distances = distances[:, min_samples - 1]
# [:, min_samples - 1]: Get last column (k-th nearest neighbor distance)
# This gives distance to k-th nearest neighbor for each point

# ============================================
# PLOTTING K-DISTANCE GRAPH
# ============================================

# Create line plot
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Plot sorted distances
plt.plot(distances)  # Line plot of k-th nearest neighbor distances
# X-axis: Points sorted by distance (implicit index)
# Y-axis: Distance to k-th nearest neighbor

# Label axes
plt.xlabel('Points sorted by distance')  # X-axis: point index (sorted)
plt.ylabel(f'{min_samples}-th Nearest Neighbor Distance')  # Y-axis: distance
plt.title('K-Distance Graph for DBSCAN')  # Chart title

# Draw horizontal line at current eps
plt.axhline(y=0.3, color='r', linestyle='--', label='eps=0.3')
# Shows where current eps value is
# Points above this line are likely noise (far from neighbors)

plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# ============================================
# FINDING OPTIMAL EPS: Knee Point Detection
# ============================================

# The "knee" (elbow) in the graph is a good eps value
# It's where the curve changes from steep to flat
# This separates dense regions (clusters) from sparse regions (noise)

from scipy.spatial.distance import cdist  # Distance computations
from scipy.stats import zscore  # Z-score for outlier detection

# Use z-score to find outliers in distances (potential knee point)
z_scores = np.abs(zscore(distances))
# zscore(): Standardizes distances (mean=0, std=1)
# np.abs(): Absolute value (we care about distance from mean, not direction)
# High z-score = point is far from mean (potential knee)

# Find first point significantly above mean (z-score > 2)
knee_idx = np.argmax(z_scores > 2)
# np.argmax(): Index of first True value
# z_scores > 2: Boolean array (True for outliers)
# This finds the "knee" where distances jump up

# Get optimal eps from knee point
optimal_eps = distances[knee_idx] if knee_idx > 0 else distances[len(distances) // 4]
# If knee found, use that distance; otherwise use 25th percentile

print(f"Suggested eps (from k-distance graph): {optimal_eps:.3f}")  # Suggested value
print(f"Current eps: {dbscan.eps}")  # Current value

# Interpretation:
# - K-distance graph shows density structure
# - Knee point separates dense (clusters) from sparse (noise)
# - eps should be around the knee point
# - Too small eps = everything is noise
# - Too large eps = everything is one cluster


## Comparing Different eps Values

Let's see how eps affects clustering results.


In [ ]:
# ============================================
# COMPARING DIFFERENT EPS VALUES: Understanding Parameter Sensitivity
# ============================================

# eps is the most critical DBSCAN parameter
# We'll test different values to see how they affect clustering results
# This helps understand the trade-off between cluster size and noise detection

# Test different eps values
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5]  # Range from small to large
# Smaller eps = tighter clusters, more noise
# Larger eps = looser clusters, less noise

# Create figure with one subplot per eps value
fig, axes = plt.subplots(1, len(eps_values), figsize=(20, 4))
# 1 row, len(eps_values) columns (5 subplots side by side)

# Test each eps value
for idx, eps in enumerate(eps_values):
    # Create DBSCAN with this eps value
    dbscan_test = DBSCAN(eps=eps, min_samples=10)  # Same min_samples for all
    
    # Get cluster assignments
    labels_test = dbscan_test.fit_predict(X)  # Cluster labels for all samples
    
    # Count clusters and noise
    n_clusters_test = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    # Number of clusters (excluding noise label -1)
    n_noise_test = list(labels_test).count(-1)  # Number of noise points
    
    # Get unique cluster labels (including -1 for noise)
    unique_labels = set(labels_test)  # Set of unique labels
    
    # Create colors for each cluster
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    # Spectral colormap: rainbow colors
    # np.linspace(0, 1, len(unique_labels)): Evenly spaced values
    
    # Plot each cluster separately
    for k, col in zip(unique_labels, colors):
        # k: Cluster label (0, 1, 2, ... or -1 for noise)
        # col: Color for this cluster
        
        if k == -1:
            # Noise points: special handling
            col = 'black'  # Black color
            marker = 'x'  # X marker
        else:
            # Regular clusters
            marker = 'o'  # Circle marker
        
        # Create boolean mask for samples in this cluster
        class_member_mask = (labels_test == k)  # True for samples in cluster k
        
        # Get coordinates of samples in this cluster
        xy = X[class_member_mask]  # Feature values for samples in cluster k
        
        # Scatter plot for this cluster
        axes[idx].scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=30, alpha=0.7)
        # xy[:, 0]: First feature, xy[:, 1]: Second feature
        # s=30: Smaller point size (many subplots)
    
    # Label this subplot
    axes[idx].set_title(f'eps={eps}\nClusters: {n_clusters_test}, Noise: {n_noise_test}')
    # Title shows eps value, number of clusters, and noise points
    axes[idx].set_xlabel('Feature 1')  # X-axis label
    axes[idx].set_ylabel('Feature 2')  # Y-axis label
    axes[idx].grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display all plots

# ============================================
# SUMMARY: How eps Affects Clustering
# ============================================

# Print summary table
print("\nSummary of eps values:")
for eps in eps_values:
    # Re-run DBSCAN for summary (could reuse previous results, but this is clearer)
    dbscan_test = DBSCAN(eps=eps, min_samples=10)
    labels_test = dbscan_test.fit_predict(X)
    
    # Count clusters and noise
    n_clusters_test = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    n_noise_test = list(labels_test).count(-1)
    
    print(f"  eps={eps}: {n_clusters_test} clusters, {n_noise_test} noise points")

# Interpretation:
# - Very small eps (0.1): Many small clusters, many noise points
# - Small eps (0.2): Fewer clusters, still many noise points
# - Medium eps (0.3): Good balance (usually optimal)
# - Large eps (0.4-0.5): Fewer clusters, less noise (may merge clusters)
# - Choose eps that gives reasonable number of clusters and noise


## Validation & Testing

Let's validate the clustering and compare with K-Means.


In [ ]:
# ============================================
# VALIDATION 1: Evaluating Clustering Quality
# ============================================

# Evaluate clustering quality using silhouette score
# Note: Silhouette score requires at least 2 clusters
# If DBSCAN finds only 1 cluster or all noise, we can't calculate silhouette

if n_clusters >= 2:
    # evaluate_clustering() calculates quality metrics
    evaluation = evaluate_clustering(X, y_pred, algorithm='DBSCAN')
    # X: Feature data
    # y_pred: Cluster assignments
    # algorithm='DBSCAN': Specify algorithm (for reporting)
    # Returns dictionary with evaluation metrics
    
    print("Clustering Evaluation:")
    print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")  # Quality metric (-1 to 1)
    print(f"  Number of clusters: {evaluation['n_clusters']}")  # Number of clusters found
    print(f"  Number of noise points: {n_noise}")  # Number of outliers
else:
    # Can't calculate silhouette with < 2 clusters
    print(f"Warning: Only {n_clusters} cluster(s) found. Silhouette score requires at least 2 clusters.")
    print(f"  Number of clusters: {n_clusters}")
    print(f"  Number of noise points: {n_noise}")

# ============================================
# VALIDATION 2: Comparing DBSCAN with K-Means
# ============================================

# DBSCAN and K-Means are very different algorithms
# DBSCAN: Density-based, handles non-spherical clusters, identifies noise
# K-Means: Partition-based, assumes spherical clusters, no noise detection
# We'll compare them on the same data to see the differences

from sklearn.cluster import KMeans  # K-Means clustering

# Create K-Means with 2 clusters (we know there are 2 true clusters)
kmeans = KMeans(n_clusters=2, random_state=42)
# n_clusters=2: Force 2 clusters (K-Means requires specifying k)

# Get K-Means cluster assignments
y_kmeans = kmeans.fit_predict(X)  # Cluster labels (0 or 1)

# ============================================
# VISUALIZING COMPARISON: DBSCAN vs K-Means
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Width=12 inches, height=5 inches

# Subplot 1: DBSCAN Results
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot colored by DBSCAN clusters
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
# c=y_pred: Color by cluster assignment (including -1 for noise)

# Highlight noise points separately
noise_mask = (y_pred == -1)  # Boolean mask: True for noise points
if np.any(noise_mask):
    # If there are noise points, plot them with special marker
    plt.scatter(X[noise_mask, 0], X[noise_mask, 1], c='black', marker='x', s=50, label='Noise')
    # Black X markers for noise points

plt.title('DBSCAN (handles non-spherical)')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: K-Means Results
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot colored by K-Means clusters
plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, cmap='viridis', s=50, alpha=0.7)
# c=y_kmeans: Color by cluster assignment (0 or 1)

# Plot cluster centers (centroids)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
# Red X marks show cluster centers
# kmeans.cluster_centers_: Coordinates of centroids

plt.title('K-Means (assumes spherical)')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# COMPARISON SUMMARY
# ============================================

print("\nComparison:")
print(f"  DBSCAN: {n_clusters} clusters, {n_noise} noise points")
# DBSCAN found clusters automatically and identified noise
print(f"  K-Means: 2 clusters (forced), 0 noise points")
# K-Means requires specifying k and doesn't identify noise
print("  Note: DBSCAN can handle non-spherical clusters, K-Means cannot")
# DBSCAN found the crescent shapes, K-Means tried to split them with a line

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check that DBSCAN found at least one cluster
assert n_clusters > 0, "Should find at least one cluster"
# If no clusters found, something went wrong (eps too small?)

print("\n✓ Validation checks passed")  # All checks passed!

# Interpretation:
# - DBSCAN: Found 2 crescent-shaped clusters, identified noise points
# - K-Means: Forced 2 clusters, split data with straight line (doesn't match true structure)
# - For non-spherical data, DBSCAN is clearly superior
# - K-Means works well for spherical clusters, but struggles with complex shapes


## Real-World Application

Let's apply DBSCAN to different types of datasets.


In [ ]:
# Test on different datasets
datasets = {
    'Circles': make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42),
    'Blobs': make_blobs(n_samples=300, centers=4, n_features=2, random_state=42, cluster_std=0.60)
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, (name, (X_data, y_true_data)) in enumerate(datasets.items()):
    # Apply DBSCAN
    dbscan_data = DBSCAN(eps=0.3, min_samples=10)
    y_pred_data = dbscan_data.fit_predict(X_data)
    
    n_clusters_data = len(set(y_pred_data)) - (1 if -1 in y_pred_data else 0)
    n_noise_data = list(y_pred_data).count(-1)
    
    # Plot true labels
    axes[idx, 0].scatter(X_data[:, 0], X_data[:, 1], c=y_true_data, cmap='viridis', s=50, alpha=0.7)
    axes[idx, 0].set_title(f'{name} - True Labels')
    axes[idx, 0].set_xlabel('Feature 1')
    axes[idx, 0].set_ylabel('Feature 2')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Plot DBSCAN results
    unique_labels = set(y_pred_data)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    for k, col in zip(unique_labels, colors):
        if k == -1:
            col = 'black'
            marker = 'x'
        else:
            marker = 'o'
        
        class_member_mask = (y_pred_data == k)
        xy = X_data[class_member_mask]
        axes[idx, 1].scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=50, alpha=0.7)
    
    axes[idx, 1].set_title(f'{name} - DBSCAN\nClusters: {n_clusters_data}, Noise: {n_noise_data}')
    axes[idx, 1].set_xlabel('Feature 1')
    axes[idx, 1].set_ylabel('Feature 2')
    axes[idx, 1].grid(True, alpha=0.3)
    
    print(f"{name}: {n_clusters_data} clusters, {n_noise_data} noise points")

plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **DBSCAN Basics**
   - Density-based clustering algorithm
   - Groups points that are closely packed
   - Automatically identifies noise/outliers
   - No need to specify number of clusters

2. **Point Types**
   - **Core Point**: Has enough neighbors within eps
   - **Border Point**: Reachable from core but not core itself
   - **Noise Point**: Neither core nor border (outlier)

3. **Key Parameters**
   - **eps**: Maximum distance for neighbors (most critical)
   - **min_samples**: Minimum neighbors to be core point
   - Use k-distance graph to find optimal eps

4. **Best Practices**
   - Use k-distance graph to choose eps
   - Start with min_samples = 2 * dimensions
   - Scale features before clustering
   - Visualize results to validate
   - Handle noise points appropriately

### When to Use DBSCAN

✅ **Good for:**
- Unknown number of clusters
- Non-spherical cluster shapes
- Need to identify outliers/noise
- Clusters of varying density
- Spatial/geographic data
- When K-Means fails (non-spherical)

❌ **Not ideal for:**
- High-dimensional data (curse of dimensionality)
- Clusters with very different densities
- When all points should be clustered (no noise)
- Very large datasets (can be slow)
- When clusters are well-separated spheres (K-Means better)

### Next Steps

- Try **HDBSCAN** (Hierarchical DBSCAN) for varying densities
- Compare with **K-Means** and **Hierarchical Clustering**
- Use **OPTICS** for automatic parameter selection
- Apply to **anomaly detection** problems
- Use for **image segmentation** tasks
